In [1]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# 01c_data_quality_failure_invest.py
# Purpose of Script: Investigate Data Quality Failures for Report
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initialization ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Google Drive
#~~~~~~~~~~~~~~~~~~~~~~~~~~
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Libraries
#~~~~~~~~~~~~~~~~~~~~~~~~~~
import numpy as np
import pandas as pd
import gc
import duckdb

In [3]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Print Versions
#~~~~~~~~~~~~~~~~~~~~~~~~~~
print(f"Numpy version = {np.__version__}")
print(f"Pandas version = {pd.__version__}")
print(f"DuckDB version = {duckdb.__version__}")

Numpy version = 2.0.2
Pandas version = 2.2.2
DuckDB version = 1.3.2


In [4]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initiate Duck Connection
#~~~~~~~~~~~~~~~~~~~~~~~~~~
con = duckdb.connect()

#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Define Input/Output Paths
#~~~~~~~~~~~~~~~~~~~~~~~~~~
### Input
path_samp = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/03_samples/"
path_rq1 = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/05_duplicates/07_rq1/"
path_out = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/03_outputs/"

In [5]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Data Quality Analysis - Raw SOR DQ ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Data Quality Results Per File
df = con.execute(f""" select * from '{path_samp}dq_final.parquet'""").df()

In [6]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Check UUID
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# 3 files exhibit problems with UUID
df_uuid = df[df["check_uuids"] == "Y"]

### Facebook - 14 entries duplicated - extracted
### Instagram - 14 entries duplicated - extracted
### Tiktok - 4 entries duplicated - extracted

In [7]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Check Platform UUID
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# These belong to the platform to assign - not to an indivdual piece of content
# Decision - Retain in Sample
df_plat_uuid = df[df["check_plat_uuids"] == "Y"]

In [8]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Extracted Duplicates ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_facebook = con.execute(f""" select * from '{path_rq1}facebook_rq1.parquet'""").df()
df_instagram = con.execute(f""" select * from '{path_rq1}instagram_rq1.parquet'""").df()
df_tiktok = con.execute(f""" select * from '{path_rq1}tiktok_rq1.parquet'""").df()
df = pd.concat([df_facebook, df_instagram, df_tiktok])

In [9]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Export Duplicates ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df.to_csv(f"{path_out}01c_dq_1_duplicate_entries.csv")